# PyBaMM SEI Formulations Sensitivity Analysis
This notebook simulates calendar ageing followed by fast cycling. It sweeps through the 4 available SEI mechanism models in PyBaMM.

In [ ]:
!pip install pybamm pandas matplotlib

In [ ]:
import pybamm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pybamm.set_logging_level("INFO")

## Base Configuration
We set up an identical `options` dictionary for a DFN model but sweep the `SEI` setting. Because of the computational complexity, we'll keep cycles at 100 to reduce execution load on standard Colab instances (You can scale `TOTAL_CYCLES` to 200 simply by modifying the variable!).

In [ ]:
STORAGE_DAYS = 30
INITIAL_SOC = 0.9
TOTAL_CYCLES = 100
CHUNK_SIZE = 10

SEI_MODELS = [
    "reaction limited",
    "solvent-diffusion limited",
    "electron-migration limited",
    "interstitial-diffusion limited"
]

EXPERIMENT_STEP = (
    "Discharge at C/9 until 3.2 V",
    "Rest for 15 minutes",
    "Charge at C/2 until 4.1 V",
    "Hold at 4.1 V until C/20",
    "Rest for 15 minutes",
)

def get_submesh_types():
    model = pybamm.lithium_ion.DFN()
    return model.default_submesh_types.copy()

var_pts = {"x_n": 30, "x_s": 30, "x_p": 30, "r_n": 50, "r_p": 50}

## Main Execution Framework
Here we loop over the 4 mechanisms, aging the cell for 30 days and then recursively looping through chunks of cycling.

In [ ]:
results_data = {}

for sei_model in SEI_MODELS:
    print(f"\n{'='*50}")
    print(f"Running SEI Model: {sei_model}")
    print(f"{'='*50}\n")
    
    options = {
        "SEI": sei_model,
        "SEI porosity change": "true",
        "lithium plating": "partially reversible",
        "lithium plating porosity change": "true",
        "particle mechanics": ("swelling and cracking", "swelling only"),
        "SEI on cracks": "true",
        "loss of active material": "stress-driven",
    }
    
    model = pybamm.lithium_ion.DFN(options)
    parameter_values = pybamm.ParameterValues("OKane2022")
    parameter_values["Current function [A]"] = 0
    solver = pybamm.IDAKLUSolver(atol=1e-6, rtol=1e-6)
    
    # Ageing
    print(f"-> Storage Phase ({STORAGE_DAYS} days)")
    seconds = STORAGE_DAYS * 24 * 60 * 60
    t_eval = np.linspace(0, seconds, min(100, max(20, STORAGE_DAYS * 2)))
    sim = pybamm.Simulation(
        model, parameter_values=parameter_values, solver=solver, var_pts=var_pts, submesh_types=get_submesh_types()
    )
    sol_ageing = sim.solve(t_eval=t_eval, initial_soc=INITIAL_SOC)
    
    # Cycling
    print(f"-> Cycling Phase ({TOTAL_CYCLES} cycles)")
    num_chunks = TOTAL_CYCLES // CHUNK_SIZE
    experiment_chunk = pybamm.Experiment([EXPERIMENT_STEP] * CHUNK_SIZE)
    
    cc_caps, cv_caps, cycles = [], [], []
    current_solution = sol_ageing
    
    # Update current for experiment
    parameter_values_cyc = pybamm.ParameterValues("OKane2022")
    
    for chunk_idx in range(num_chunks):
        print(f"   Chunk {chunk_idx + 1}/{num_chunks}... ", end="")
        sim_cyc = pybamm.Simulation(
            model,
            experiment=experiment_chunk,
            parameter_values=parameter_values_cyc,
            solver=solver,
            var_pts=var_pts,
            submesh_types=get_submesh_types(),
        )
        
        chunk_solution = sim_cyc.solve(starting_solution=current_solution)
        
        # Parse data
        for cycle_sol in chunk_solution.cycles:
            steps = cycle_sol.steps
            charge_c_step = steps[2] if len(steps) > 2 else None
            charge_v_step = steps[3] if len(steps) > 3 else None
            
            c_cap = charge_c_step["Capacity [A.h]"].entries[-1] - charge_c_step["Capacity [A.h]"].entries[0] if charge_c_step else 0.0
            v_cap = charge_v_step["Capacity [A.h]"].entries[-1] - charge_v_step["Capacity [A.h]"].entries[0] if charge_v_step else 0.0
            
            cc_caps.append(c_cap)
            cv_caps.append(v_cap)
            cycles.append(len(cycles) + 1)
            
        current_solution = chunk_solution
        print("Done.")

    results_data[sei_model] = {
        "cycles": cycles,
        "cc_caps": cc_caps,
        "cv_caps": cv_caps
    }

## Analysis and Ratios
We will construct a Pandas table explicitly filtering for milestone capacities and rendering the CC/CV ratios.

In [ ]:
# Build Pandas DF
rows = []
for model_name, data in results_data.items():
    for c, cc, cv in zip(data["cycles"], data["cc_caps"], data["cv_caps"]):
        if c in [10, 50, 100, 150, 200]:
            rows.append({
                "Variant": model_name,
                "Cycle": c,
                "CC Capacity [A.h]": cc,
                "CV Capacity [A.h]": cv,
                "CC/CV Ratio": cc / cv if cv > 0 else float('nan')
            })

df = pd.DataFrame(rows)
pivot_cc = df.pivot(index='Cycle', columns='Variant', values='CC Capacity [A.h]')
pivot_cv = df.pivot(index='Cycle', columns='Variant', values='CV Capacity [A.h]')
pivot_ratio = df.pivot(index='Cycle', columns='Variant', values='CC/CV Ratio')

print("--- CC Capacity [A.h] ---")
print(pivot_cc.round(3))
print("\n--- CV Capacity [A.h] ---")
print(pivot_cv.round(3))
print("\n--- CC/CV Ratio ---")
print(pivot_ratio.round(2))

## Visual Plots
A cleanly separated view of the internal capacities for visual verification directly inside this notebook.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 6))

for model_name, data in results_data.items():
    axs[0].plot(data["cycles"], data["cc_caps"], marker=".", label=model_name)
    axs[1].plot(data["cycles"], data["cv_caps"], marker=".", label=model_name)

axs[0].set_title("CC Capacity Decay")
axs[0].set_ylabel("Capacity [A.h]")
axs[0].set_xlabel("Cycles")
axs[0].grid(True)
axs[0].legend()

axs[1].set_title("CV Capacity Reliance")
axs[1].set_ylabel("Capacity [A.h]")
axs[1].set_xlabel("Cycles")
axs[1].grid(True)
axs[1].legend()

plt.tight_layout()
plt.show()